In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/hauuto/catbreed-id.git"
BRANCH = "main"

WORK_DIR = Path("/kaggle/working/catbreed-id")
OUTPUTS_DIR = Path("/kaggle/working/outputs")
HF_REPO_ID = "catbreed-id/catbreed-experiments"

In [ ]:
import subprocess
import sys

if not WORK_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(WORK_DIR)],
        check=True,
    )
else:
    changes = subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=WORK_DIR,
        text=True,
    ).strip()

    if changes:
        raise RuntimeError(
            "Repo có thay đổi chưa commit. Kiểm tra trước khi cập nhật."
        )

    subprocess.run(
        ["git", "switch", BRANCH],
        cwd=WORK_DIR,
        check=True,
    )
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=WORK_DIR,
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-r", str(WORK_DIR / "requirements.txt"),
    ],
    check=True,
)

GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=WORK_DIR,
    text=True,
).strip()

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Code sẵn sàng:", GIT_COMMIT)

In [ ]:
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

api = HfApi(token=os.environ["HF_TOKEN"])
RUN_OWNER = api.whoami()["name"]

# Xác nhận tài khoản truy cập được kho kết quả.
api.repo_info(repo_id=HF_REPO_ID, repo_type="dataset")

print("Tài khoản:", RUN_OWNER)
print("Kho kết quả:", HF_REPO_ID)
print("Runner sẵn sàng")

In [ ]:
import importlib
import json
from datetime import datetime, timezone
from uuid import uuid4

import torch
import yaml

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

importlib.invalidate_caches()
from src.training.artifacts import save_stage, sync_results

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    + "_smoke_"
    + uuid4().hex[:8]
)
run_dir = OUTPUTS_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=False)

info = {
    "run_id": run_id,
    "owner": RUN_OWNER,
    "git_commit": GIT_COMMIT,
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Kiểm tra workflow, không huấn luyện",
    "is_smoke_test": True,
    "data_version": None,
    "split_id": None,
    "training_seed": None,
    "status": "running",
}

def write_info():
    (run_dir / "info.json").write_text(
        json.dumps(info, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

write_info()

(run_dir / "config.yaml").write_text(
    yaml.safe_dump({
        "mode": "smoke",
        "stages": ["frozen", "finetuned"],
        "hf_repo_id": HF_REPO_ID,
    }),
    encoding="utf-8",
)

(run_dir / "history.csv").write_text(
    "stage,event\nfrozen,smoke_checkpoint\nfinetuned,smoke_checkpoint\n",
    encoding="utf-8",
)

(run_dir / "requirements_freeze.txt").write_text(
    subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    ),
    encoding="utf-8",
)

sync_status = {}

for stage, value in [("frozen", 1.0), ("finetuned", 2.0)]:
    sync_status[stage] = save_stage(
        run_dir=run_dir,
        stage=stage,
        checkpoint={
            "is_smoke_test": True,
            "tensor": torch.tensor([value]),
        },
        metrics={"is_smoke_test": True, "stage": stage},
        repo_id=HF_REPO_ID,
    )

info["status"] = "completed"
info["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
write_info()

sync_status["final"] = sync_results(run_dir, HF_REPO_ID)

print("\nRun ID:", run_id)
for step, success in sync_status.items():
    print(f"{step}: {'Đã đồng bộ' if success else 'Chưa đồng bộ'}")

if not all(sync_status.values()):
    raise RuntimeError(
        f"Có bước chưa đồng bộ. Giữ thư mục để upload lại: {run_dir}"
    )

print("Workflow thử hoàn thành")

## Cấu hình mô hình và thực nghiệm


In [ ]:
CONFIG = {
    "profile": "smoke",  # smoke: kiểm tra ngắn; official: huấn luyện đầy đủ
    "protocol_confirmed": True,
    "protocol_id": "Nhom09_QD1-QD10_2026-10-18",
    "protocol_source_sha256": "85d6889b4a5069f6b92211887c800dd57ec84c33ea58da3746ff94335bd93726",
    "split_policy": "official_test_trainval_80_20",
    "split_seed": 42,
    "expected_split_id": "3b283d422973d497d268d935ef0fe4a76bbd2d10f4b5efdef20a8d859949cfcb",
    "models": ["resnet50", "mobilenet_v2", "efficientnet_b0"],
    "training_seeds": [42],  # Tuần 10; seed 43/44 và baseline theo lịch tuần 11–12
    "owner": "Tai",
    "data_input_root": "/kaggle/input",
    "output_root": "/kaggle/working/outputs",
    "far_ood_root": "/kaggle/input",  # DTD chính thức, chỉ dùng đánh giá cuối
    "image_size": 224,
    "batch_size": 32,
    "workers": 0,
    "progress_interval_seconds": 15,
    "command_timeout_seconds": 300,
    "sync_timeout_seconds": 600,
    "download_timeout_seconds": 600,
    "network_read_timeout_seconds": 30,
    "pretrained": True,
    "augmentation": True,
    "frozen_epochs": 10,
    "finetuned_epochs": 20,
    "baseline_epochs": 30,
    "frozen_lr": 1e-3,
    "head_lr": 1e-4,
    "backbone_lr": 1e-5,
    "weight_decay": 1e-4,
    "label_smoothing": 0.0,
    "dropout": 0.3,
    "early_stopping_patience": 7,
    "scheduler_patience": 3,
    "fine_tune_blocks": 2,
    "knn_k": 10,
    "energy_temperature": 1.0,
    "ece_bins": 15,
    "evaluation_locked": False,
    "run_final_evaluation": False,
    "run_background_ablation": False,
    "run_head_ablation": False,
    "run_ood_sensitivity": False,
    "bootstrap_repeats": 2000,
    "bootstrap_seed": 42,
    "hf_sync": True,
    "hf_repo_id": "catbreed-id/catbreed-experiments",
    "smoke_samples_per_class": 2,
    "smoke_batches": 2,
}
SUBMISSION = {}  # Metadata nguồn của lượt chạy, không chứa token
AUTO_RUN = True


## Theo dõi tiến trình và thời gian chờ


In [ ]:
import sys
import json
import time
import threading
import subprocess
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from itertools import islice
from tqdm import tqdm

_LOG_LOCK = threading.Lock()


def log_event(step, status="info", **details):
    record = {
        "time": datetime.now(timezone.utc).isoformat(),
        "step": step,
        "status": status,
        **details,
    }
    line = json.dumps(record, ensure_ascii=False, allow_nan=False)
    with _LOG_LOCK:
        tqdm.write(line, file=sys.stdout)
        sys.stdout.flush()
        root = Path(CONFIG["output_root"])
        if root.is_absolute():
            root.mkdir(parents=True, exist_ok=True)
            with (root / "progress_events.jsonl").open("a", encoding="utf-8") as stream:
                stream.write(line + "\n")


@contextmanager
def timed_step(label):
    started = time.monotonic()
    stopped = threading.Event()
    log_event(label, "started")

    def heartbeat():
        while not stopped.wait(CONFIG["progress_interval_seconds"]):
            log_event(label, "waiting", elapsed_seconds=round(time.monotonic() - started, 1))

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    try:
        yield
    except BaseException as exc:
        log_event(label, "failed", error_type=type(exc).__name__)
        raise
    else:
        log_event(label, "completed", elapsed_seconds=round(time.monotonic() - started, 1))
    finally:
        stopped.set()
        worker.join(timeout=1)


def run_step(label, function, *args, **kwargs):
    with timed_step(label):
        return function(*args, **kwargs)


def progress(items, label, total=None):
    if total is None and hasattr(items, "__len__"):
        total = len(items)
    started = previous = time.monotonic()
    log_event(label, "started", total=total)
    with tqdm(
        total=total,
        desc=label,
        file=sys.stdout,
        ascii=True,
        ncols=110,
        mininterval=2,
        miniters=1,
        leave=True,
    ) as bar:
        for completed, item in enumerate(items, 1):
            yield bar, item
            bar.update(1)
            now = time.monotonic()
            if (
                completed == 1
                or completed == total
                or now - previous >= CONFIG["progress_interval_seconds"]
            ):
                elapsed = now - started
                remaining = elapsed / completed * (total - completed) if total is not None else None
                log_event(
                    label,
                    "progress",
                    completed=completed,
                    total=total,
                    elapsed_seconds=round(elapsed, 1),
                    eta_seconds=round(remaining, 1) if remaining is not None else None,
                    metrics=bar.postfix,
                )
                previous = now


def run_command(arguments, label, capture=False, timeout=None):
    timeout = CONFIG["command_timeout_seconds"] if timeout is None else timeout
    with timed_step(label):
        return subprocess.run(
            arguments, check=True, text=True, capture_output=capture, timeout=timeout
        )


## Chuẩn bị thư viện và thiết bị tính toán


In [ ]:
log_event("Chuẩn bị thư viện", "started")

import importlib.util
import os
import sys
import subprocess

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
ON_KAGGLE = os.path.isdir("/kaggle/working")
if ON_KAGGLE:
    for module, package in [
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("sklearn", "scikit-learn"),
        ("scipy", "scipy"),
        ("matplotlib", "matplotlib"),
        ("huggingface_hub", "huggingface_hub"),
    ]:
        if importlib.util.find_spec(module) is None:
            run_command([sys.executable, "-m", "pip", "install", package], "Cài " + package)

import hashlib
import json
import math
from pathlib import Path
import random
import time
import traceback
from datetime import datetime, timezone
from uuid import uuid4
from xml.etree import ElementTree as ET
import numpy as np
import pandas as pd
from PIL import Image
from scipy.special import logsumexp, softmax
from scipy.optimize import minimize_scalar
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(
    "Python",
    sys.version.split()[0],
    "| torch",
    torch.__version__,
    "| torchvision",
    torchvision.__version__,
    flush=True,
)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", flush=True)

log_event("Chuẩn bị thư viện", "completed", cuda=torch.cuda.is_available())


## Nối pipeline bổ sung với kết quả của bốn cell gốc


In [ ]:
GROUP_SAVE_STAGE = None
GROUP_SYNC_RESULTS = None
RUNNER_PROVENANCE = {"used_group_runner": False, "reason": "Chưa nối với 4 cell gốc"}


def prepare_group_runner():
    global GROUP_SAVE_STAGE, GROUP_SYNC_RESULTS, RUNNER_PROVENANCE
    if not ON_KAGGLE:
        print("Kiểm chứng local: không chạy 4 cell kết nối gốc hoặc upload HF.", flush=True)
        return
    if not all(
        name in globals() for name in ["GIT_COMMIT", "RUN_OWNER", "save_stage", "sync_results"]
    ):
        raise RuntimeError("Chạy đủ 4 cell gốc phía trên trước khi chạy phần bổ sung.")
    if CONFIG["hf_repo_id"] != HF_REPO_ID:
        raise RuntimeError("Kho HF trong CONFIG phải khớp HF_REPO_ID của cell gốc.")
    changes = run_command(
        ["git", "-C", str(WORK_DIR), "status", "--porcelain"], "Kiểm tra Git", capture=True
    ).stdout.strip()
    if changes:
        raise RuntimeError("Repo có thay đổi chưa commit. Kiểm tra trước khi đổi phiên bản code.")
    actual_commit = run_command(
        ["git", "-C", str(WORK_DIR), "rev-parse", "HEAD"], "Đọc commit", capture=True
    ).stdout.strip()
    requested = SUBMISSION.get("git_commit")
    if requested and requested != actual_commit:
        run_command(
            ["git", "-C", str(WORK_DIR), "fetch", "origin", requested], "Lấy commit đã chốt"
        )
        run_command(
            ["git", "-C", str(WORK_DIR), "checkout", "--detach", requested], "Chọn commit đã chốt"
        )
        actual_commit = run_command(
            ["git", "-C", str(WORK_DIR), "rev-parse", "HEAD"], "Xác nhận commit", capture=True
        ).stdout.strip()
    notebook_at_commit = WORK_DIR / "notebooks/kaggle_runner.ipynb"
    matches = False
    if notebook_at_commit.is_file() and SUBMISSION.get("notebook_source_sha256"):
        matches = (
            notebook_source_hash(json.loads(notebook_at_commit.read_text(encoding="utf-8")))
            == SUBMISSION["notebook_source_sha256"]
        )
    if CONFIG["profile"] == "official" and not matches:
        raise RuntimeError(
            "Source notebook chưa khớp commit trên GitHub. Push nhánh đã commit rồi cập nhật notebook trước khi train."
        )
    import importlib

    importlib.invalidate_caches()
    artifacts = importlib.import_module("src.training.artifacts")
    if Path(artifacts.__file__).resolve() != (WORK_DIR / "src/training/artifacts.py").resolve():
        raise RuntimeError("Module src được nạp từ đường dẫn khác; restart session rồi chạy lại.")
    artifacts = importlib.reload(artifacts)

    def bounded_sync(run_dir, repo_id=HF_REPO_ID):
        try:
            run_command(
                [
                    sys.executable,
                    str(WORK_DIR / "scripts/sync_run.py"),
                    "--run-dir",
                    str(Path(run_dir).resolve()),
                    "--repo-id",
                    repo_id,
                ],
                "Đồng bộ Hugging Face",
                timeout=CONFIG["sync_timeout_seconds"],
            )
            return True
        except (subprocess.SubprocessError, OSError) as exc:
            log_event(
                "Đồng bộ Hugging Face",
                "failed",
                error_type=type(exc).__name__,
                output_retained=True,
            )
            return False

    artifacts.sync_results = bounded_sync
    GROUP_SAVE_STAGE, GROUP_SYNC_RESULTS = artifacts.save_stage, artifacts.sync_results
    RUNNER_PROVENANCE = {
        "used_group_runner": True,
        "bootstrap_git_commit": GIT_COMMIT,
        "bootstrap_owner": RUN_OWNER,
        "runner_git_commit": actual_commit,
        "runner_repo": REPO_URL,
        "sync_timeout_seconds": CONFIG["sync_timeout_seconds"],
        "notebook_source_matches_commit": matches,
        "artifact_module_sha256": hashlib.sha256(
            (WORK_DIR / "src/training/artifacts.py").read_bytes()
        ).hexdigest(),
        "sync_script_sha256": hashlib.sha256(
            (WORK_DIR / "scripts/sync_run.py").read_bytes()
        ).hexdigest(),
    }
    print("Đã nối phần bổ sung với runner nhóm:", actual_commit, flush=True)


## Khai báo 12 giống mèo và các hàm đọc ảnh, ghi metadata


In [ ]:
CAT_BREEDS = [
    "Abyssinian",
    "Bengal",
    "Birman",
    "Bombay",
    "British_Shorthair",
    "Egyptian_Mau",
    "Maine_Coon",
    "Persian",
    "Ragdoll",
    "Russian_Blue",
    "Siamese",
    "Sphynx",
]

CLASS_TO_IDX = {name: index for index, name in enumerate(CAT_BREEDS)}

EXPECTED_CAT_COUNTS = dict(
    zip(CAT_BREEDS, [198, 200, 200, 184, 200, 190, 200, 200, 200, 200, 199, 200])
)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
    )
    temporary.replace(path)


def file_hash(path, algorithm="sha256"):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, algorithm).hexdigest()


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def read_rgb(path):
    # Đọc header thực tế, kể cả PNG mang đuôi .jpg. Alpha ghép nền xám có quy định rõ.
    with Image.open(path) as source:
        source.load()
        if source.mode in ["RGBA", "LA"] or "transparency" in source.info:
            rgba = source.convert("RGBA")
            return Image.alpha_composite(
                Image.new("RGBA", rgba.size, (128, 128, 128, 255)), rgba
            ).convert("RGB")
        return source.convert("RGB")


def notebook_source_hash(notebook):
    content = [(c["cell_type"], "".join(c["source"])) for c in notebook["cells"]]
    return hashlib.sha256(json.dumps(content, ensure_ascii=False).encode()).hexdigest()


## Đồng bộ bằng kết nối Hugging Face đã xác thực


In [ ]:
def get_hf_api(required=False):
    if ON_KAGGLE:
        if not CONFIG["hf_sync"] or not os.environ.get("HF_TOKEN"):
            raise RuntimeError("Pipeline Kaggle cần HF_TOKEN và hf_sync=True.")
        CONFIG["hf_authenticated_owner"] = RUN_OWNER
        return api  # Đã xác thực ở cell gốc thứ 3.
    if required:
        raise RuntimeError("Huấn luyện chính thức cần chạy trên Kaggle với 4 cell gốc.")
    return None


def sync_run(run_dir, api, step):
    if api is None:
        print(step + ": kiểm chứng local, chưa đồng bộ", flush=True)
        return False
    if GROUP_SYNC_RESULTS is None:
        raise RuntimeError("Chưa nối hàm đồng bộ của nhóm.")
    success = bool(GROUP_SYNC_RESULTS(run_dir, CONFIG["hf_repo_id"]))
    print(
        step + (": Đã đồng bộ" if success else ": Chưa đồng bộ; giữ output để upload lại"),
        flush=True,
    )
    return success


## Đọc dataset đã chuẩn bị và kiểm tra checksum


In [ ]:
def locate_dataset(config):
    root = Path(config["data_input_root"]).expanduser().resolve()
    candidates = [
        p.parent.parent
        for p in root.rglob("data_audit.json")
        if p.parent.name == "manifests" and (p.parent / "all_labeled_images.csv").is_file()
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            "Gắn đúng một dataset catbreed-cats-dogs-all đã kiểm tra; hoặc đặt data_input_root cụ thể."
        )
    folder = candidates[0]
    audit = json.loads((folder / "manifests/data_audit.json").read_text(encoding="utf-8"))
    if (audit["official_cats"], audit["official_dogs"], audit["total_images_preserved"]) != (
        2371,
        4978,
        7390,
    ):
        raise ValueError("Dataset không khớp bộ Oxford đã chuẩn bị.")
    checksums = json.loads((folder / "manifests/file_checksums.json").read_text(encoding="utf-8"))
    for name in [
        "annotations/list.txt",
        "annotations/trainval.txt",
        "annotations/test.txt",
        "manifests/all_labeled_images.csv",
    ]:
        if file_hash(folder / name) != checksums[name]:
            raise ValueError("Nhãn hoặc manifest sai SHA-256: " + name)
    config["resolved_images_root"] = str(folder / "images")
    config["resolved_annotations_root"] = str(folder / "annotations")
    print(
        "Dataset:",
        folder,
        "| 7.349 ảnh có nhãn; 41 ảnh ngoài danh sách không dùng train.",
        flush=True,
    )
    return folder, {
        "source": "roonyvtt/catbreed-cats-dogs-all",
        "source_archives": audit["source_archives"],
        "checksum_index_sha256": file_hash(folder / "manifests/file_checksums.json"),
    }


## Đọc nhãn loài và kiểm tra ảnh, trimap, bounding box


In [ ]:
def read_annotation_list(path):
    result = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        image_id, original_class, species, breed_id = line.split()
        result.append(
            {
                "image_id": image_id,
                "original_class": int(original_class),
                "species": int(species),
                "breed_id": int(breed_id),
                "breed": image_id.rsplit("_", 1)[0],
            }
        )
    return result


def inspect_sample(row, folder, images_dir=None, annotations_dir=None):
    row = dict(row)
    images_dir = Path(images_dir) if images_dir is not None else folder / "images"
    annotations_dir = (
        Path(annotations_dir) if annotations_dir is not None else folder / "annotations"
    )
    image_path = images_dir / (row["image_id"] + ".jpg")
    image = read_rgb(image_path)
    row.update(
        image_path=str(image_path),
        width=image.width,
        height=image.height,
        image_sha256=file_hash(image_path),
        label=CLASS_TO_IDX.get(row["breed"], -1),
    )
    row.update(trimap_path="", has_valid_trimap=False, bbox_json="", has_bbox=False)
    if row["species"] == 2:
        return row  # near-OOD chỉ cần ảnh; không đọc mask/XML chó không dùng cho thí nghiệm
    mask_path = annotations_dir / "trimaps" / (row["image_id"] + ".png")
    if mask_path.exists():
        with Image.open(mask_path) as source:
            mask = np.array(source)
        row["trimap_path"] = str(mask_path)
        row["has_valid_trimap"] = bool(
            mask.shape == (image.height, image.width)
            and np.isin(mask, [1, 2, 3]).all()
            and (mask == 1).any()
        )
    xml_path = annotations_dir / "xmls" / (row["image_id"] + ".xml")
    if xml_path.exists():
        node = ET.parse(xml_path).find(".//bndbox")
        if node is not None:
            # PASCAL VOC dùng tọa độ 1-based inclusive; chuyển sang PIL 0-based exclusive.
            xmin, ymin, xmax, ymax = [
                int(node.findtext(key)) for key in ["xmin", "ymin", "xmax", "ymax"]
            ]
            box = [
                max(0, xmin - 1),
                max(0, ymin - 1),
                min(image.width, xmax),
                min(image.height, ymax),
            ]
            if box[0] < box[2] and box[1] < box[3]:
                row["bbox_json"], row["has_bbox"] = json.dumps(box), True
    return row


## Tạo một lần và đọc lại danh sách chia tập theo QĐ1


In [ ]:
def fixed_split(cats, destination, config):
    if config["split_policy"] != "official_test_trainval_80_20" or config["split_seed"] != 42:
        raise ValueError("QĐ1: giữ test chính thức, chia trainval 80/20 với seed 42.")
    path = Path(destination) / "fixed_split.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        mapping = cats[["image_id"]].copy()
        mapping["split"] = "test"
        pool = cats.index[cats.original_split == "trainval"]
        train, val = train_test_split(
            pool, test_size=0.2, stratify=cats.loc[pool, "label"], random_state=42
        )
        mapping.loc[train, "split"], mapping.loc[val, "split"] = "train", "val"
        temporary = path.with_suffix(".csv.tmp")
        mapping.sort_values("image_id").to_csv(temporary, index=False)
        temporary.replace(path)
    mapping = pd.read_csv(path).set_index("image_id", verify_integrity=True)
    if set(mapping.index) != set(cats.image_id):
        raise ValueError("Manifest chia tập không phủ đúng 2.371 ảnh mèo.")
    result = cats.copy()
    result["split"] = result.image_id.map(mapping["split"])
    if result.split.value_counts().to_dict() != {"test": 1183, "train": 950, "val": 238}:
        raise ValueError("Số ảnh không khớp QĐ1: 950 train / 238 val / 1.183 test.")
    if set(result.loc[result.split == "test", "image_id"]) != set(
        cats.loc[cats.original_split == "test", "image_id"]
    ):
        raise ValueError("Không được thay đổi tập test chính thức.")
    return result


## Kiểm tra dữ liệu và xuất manifest theo QĐ1


In [ ]:
_DATA_AUDIT_CACHE = {}


def make_manifests(folder, destination, config):
    annotations_dir = Path(config.get("resolved_annotations_root") or folder / "annotations")
    images_dir = Path(config.get("resolved_images_root") or folder / "images")
    rows = read_annotation_list(annotations_dir / "list.txt")
    prepared = pd.read_csv(folder / "manifests/all_labeled_images.csv").set_index(
        "image_id", verify_integrity=True
    )
    if set(prepared.index) != {row["image_id"] for row in rows}:
        raise ValueError("Manifest và list.txt không có cùng danh sách ảnh.")
    cache_key = (
        str(images_dir.resolve()),
        str(annotations_dir.resolve()),
        file_hash(annotations_dir / "list.txt"),
    )
    cached = _DATA_AUDIT_CACHE.get(cache_key)
    original_train = {r["image_id"] for r in read_annotation_list(annotations_dir / "trainval.txt")}
    original_test = {r["image_id"] for r in read_annotation_list(annotations_dir / "test.txt")}
    if original_train & original_test:
        raise ValueError("Official trainval/test overlap.")
    cats, dogs = [], []
    for row_number, row in enumerate(
        item for _, item in progress(rows if cached is None else [], "Kiểm tra ảnh Oxford")
    ):
        row["original_split"] = "trainval" if row["image_id"] in original_train else "test"
        if row["image_id"] not in original_train | original_test:
            raise ValueError("Ảnh list không có official split: " + row["image_id"])
        if row["species"] == 1:
            if row["breed"] not in CLASS_TO_IDX:
                raise ValueError("Giống mèo ngoài thiết kế.")
            cats.append(
                inspect_sample(row, folder, images_dir=images_dir, annotations_dir=annotations_dir)
            )
        elif row["species"] == 2:
            # Đọc/kiểm tra cả ảnh near-OOD, nhưng không đưa vào huấn luyện.
            dogs.append(
                inspect_sample(row, folder, images_dir=images_dir, annotations_dir=annotations_dir)
            )
        if (row_number + 1) % 1000 == 0:
            print("Đã kiểm tra", row_number + 1, "ảnh trong list.txt", flush=True)
    cats, dogs = (
        (pd.DataFrame(cats), pd.DataFrame(dogs))
        if cached is None
        else (cached[0].copy(), cached[1].copy())
    )
    for frame in [cats, dogs]:
        expected = prepared.loc[frame["image_id"]]
        if not (
            np.array_equal(frame.image_sha256.to_numpy(), expected.sha256.to_numpy())
            and np.array_equal(frame.species.to_numpy(), expected.species_id.to_numpy())
        ):
            raise ValueError("Ảnh hoặc nhãn loài khác manifest đã kiểm tra.")
    if cached is None:
        _DATA_AUDIT_CACHE[cache_key] = (cats.copy(), dogs.copy())
    if len(cats) != 2371 or cats["breed"].value_counts().to_dict() != EXPECTED_CAT_COUNTS:
        raise ValueError("Không khớp 2.371 ảnh/12 giống trong đề cương.")
    if len(dogs) != 4978:
        raise ValueError("Không khớp 4.978 ảnh chó chính thức.")
    if cats["image_id"].duplicated().any() or cats["image_sha256"].duplicated().any():
        raise ValueError("Phát hiện ảnh mèo lặp; cần quy tắc nhóm trước khi chia.")
    if set(cats.image_sha256) & set(dogs.image_sha256):
        raise ValueError("ID và near-OOD trùng nội dung.")
    cats = fixed_split(cats, destination, config)
    dogs["split"] = "near_ood"
    cats, dogs = cats.sort_values("image_id").reset_index(drop=True), dogs.sort_values(
        "image_id"
    ).reset_index(drop=True)
    identity = {
        "policy": config["split_policy"],
        "seed": config["split_seed"],
        "assignments": cats[["image_id", "label", "split", "image_sha256"]].to_dict("records"),
    }
    split_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    if split_id != config["expected_split_id"]:
        raise ValueError("Danh sách chia tập khác manifest QĐ1 đã cố định.")
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    cats.to_csv(destination / "cats_manifest.csv", index=False)
    dogs.to_csv(destination / "near_ood_manifest.csv", index=False)
    cats.loc[cats.has_valid_trimap].to_csv(destination / "valid_trimap_manifest.csv", index=False)
    cats.loc[cats.has_bbox].to_csv(destination / "head_bbox_manifest.csv", index=False)
    write_json(destination / "class_mapping.json", CLASS_TO_IDX)
    audit = {
        "split_id": split_id,
        "split_policy": config["split_policy"],
        "split_seed": config["split_seed"],
        "cat_count": len(cats),
        "dog_count": len(dogs),
        "fixed_split_sha256": file_hash(destination / "fixed_split.csv"),
        "split_counts": cats["split"].value_counts().to_dict(),
        "empty_or_invalid_trimaps": cats.loc[~cats.has_valid_trimap, "image_id"].tolist(),
        "cats_with_bbox": int(cats.has_bbox.sum()),
        "bbox_by_split": cats.groupby("split").has_bbox.sum().astype(int).to_dict(),
        "trimap_by_split": cats.groupby("split").has_valid_trimap.sum().astype(int).to_dict(),
    }
    write_json(destination / "data_audit.json", audit)
    print(pd.crosstab(cats.breed, cats.split).to_string(), flush=True)
    print("Split ID:", split_id[:16], flush=True)
    print(
        "Mask không dùng được:",
        len(audit["empty_or_invalid_trimaps"]),
        "| Bbox:",
        audit["cats_with_bbox"],
        flush=True,
    )
    return cats, dogs, audit


## Resize, chuẩn hóa và augmentation cho tập train


In [ ]:
def make_transform(training=False, augmentation=True, config=None):
    config = CONFIG if config is None else config
    operations = []
    if training and augmentation:
        operations += [
            transforms.RandomResizedCrop(
                config["image_size"],
                scale=(0.8, 1.0),
                ratio=(0.75, 4 / 3),
                interpolation=InterpolationMode.BILINEAR,
            ),
            transforms.RandomHorizontalFlip(0.5),
            transforms.RandomRotation(15, interpolation=InterpolationMode.BILINEAR, fill=128),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        ]
    else:
        operations += [
            transforms.Resize(
                (config["image_size"], config["image_size"]),
                interpolation=InterpolationMode.BILINEAR,
            )
        ]
    return transforms.Compose(
        operations + [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    )


## Tạo ảnh đầy đủ, ảnh xóa nền và ảnh cắt đầu


In [ ]:
def read_view(row, view="full"):
    image = read_rgb(row.image_path)
    if view == "no_background":
        if not row.has_valid_trimap:
            raise ValueError("Không dùng trimap rỗng để xóa nền.")
        with Image.open(row.trimap_path) as source:
            keep = np.array(source) != 2  # QĐ5: giữ cả nhãn 1 và nhãn 3.
        pixels = np.array(image).copy()
        pixels[~keep] = 128
        return Image.fromarray(pixels)
    if view == "head":
        if not row.has_bbox:
            raise ValueError("Thiếu bounding box vùng đầu; không tự tạo annotation.")
        x0, y0, x1, y1 = json.loads(row.bbox_json)
        dx, dy = 0.1 * (x1 - x0), 0.1 * (y1 - y0)
        box = (
            max(0, math.floor(x0 - dx)),
            max(0, math.floor(y0 - dy)),
            min(image.width, math.ceil(x1 + dx)),
            min(image.height, math.ceil(y1 + dy)),
        )
        return image.crop(box)
    if view != "full":
        raise ValueError("Biến thể dữ liệu không hợp lệ: " + view)
    return image


## Đọc ảnh đầy đủ, ảnh xóa nền hoặc vùng đầu


In [ ]:
class PetDataset(Dataset):
    def __init__(self, frame, training=False, augmentation=True, view="full"):
        self.frame = frame.reset_index(drop=True)
        self.transform = make_transform(training, augmentation)
        self.view = view

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return self.transform(read_view(row, self.view)), int(row.label), str(row.image_id)


## Tạo DataLoader và cố định seed


In [ ]:
def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)


def make_loader(frame, training=False, seed=42, view="full"):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        PetDataset(frame, training, CONFIG["augmentation"], view),
        batch_size=CONFIG["batch_size"],
        shuffle=training,
        num_workers=CONFIG["workers"],
        worker_init_fn=seed_worker,
        generator=generator,
        pin_memory=torch.cuda.is_available(),
    )


def small_stratified(frame, count):
    return (
        frame.groupby("label", group_keys=False)
        .head(count)
        .sort_values("image_id")
        .reset_index(drop=True)
    )


## Đọc và kiểm tra dataset far-OOD nếu có


In [ ]:
def make_far_manifest(config):
    root = Path(config["far_ood_root"]).expanduser()
    candidates = list(root.rglob("far_ood_manifest.csv")) if root.is_dir() else []
    if len(candidates) != 1:
        raise RuntimeError("QĐ7 cần gắn đúng một dataset DTD chính thức có manifest 5.640 ảnh.")
    path = candidates[0]
    base = path.parent.parent
    audit = json.loads((path.parent / "dtd_audit.json").read_text(encoding="utf-8"))
    if file_hash(path) != audit["manifest_sha256"]:
        raise ValueError("Manifest DTD sai SHA-256.")
    if audit["source_archive_md5"] != "fff73e5086ae6bdbea199a49dfb8a4c1":
        raise ValueError("Không phải archive DTD chính thức r1.0.1.")
    frame = pd.read_csv(path)
    counts = frame.category.value_counts()
    if (
        len(frame) != 5640
        or len(counts) != 47
        or not counts.eq(120).all()
        or frame.image_id.duplicated().any()
    ):
        raise ValueError("DTD phải có đủ 5.640 ảnh thuộc 47 nhóm, mỗi nhóm 120 ảnh.")
    for _, row in progress(frame.itertuples(), "Kiểm tra ảnh DTD", total=len(frame)):
        image = (base / row.image_path).resolve()
        if not image.is_relative_to(base.resolve()) or file_hash(image) != row.image_sha256:
            raise ValueError("Đường dẫn hoặc SHA-256 ảnh DTD sai: " + row.image_id)
    frame["image_path"] = [str(base / name) for name in frame.image_path]
    frame["image_id"] = "dtd/" + frame.image_id
    frame["source"] = audit["source"]
    frame["split"], frame["label"] = "far_ood", -1
    frame = frame.sort_values("image_id").reset_index(drop=True)
    frame.attrs["provenance"] = {
        key: audit[key]
        for key in [
            "dataset",
            "source",
            "source_archive_md5",
            "source_archive_sha256",
            "manifest_sha256",
        ]
    }
    return frame


## Đọc dữ liệu và chuẩn bị manifest dùng chung trước khi huấn luyện


In [ ]:
def prepare_data():
    prepare_group_runner()
    api = get_hf_api(required=ON_KAGGLE)
    root = Path(CONFIG["output_root"])
    root.mkdir(parents=True, exist_ok=True)
    folder, provenance = run_step("Tìm dataset Oxford", locate_dataset, CONFIG)
    cats, dogs, audit = run_step(
        "Kiểm tra Oxford và cố định split", make_manifests, folder, root / "data_manifest", CONFIG
    )
    frames = {split: cats[cats.split == split].copy() for split in ["train", "val", "test"]}
    frames["near_ood"] = dogs
    far = run_step("Kiểm tra DTD", make_far_manifest, CONFIG)
    if set(far.image_sha256) & set(pd.concat([cats, dogs]).image_sha256):
        raise ValueError("DTD trùng nội dung ảnh Oxford.")
    frames["far_ood"] = far
    provenance["far_ood"] = far.attrs["provenance"]
    far.to_csv(root / "data_manifest/far_ood_manifest.csv", index=False)
    if CONFIG["profile"] == "smoke":
        for split in ["train", "val", "test"]:
            frames[split] = small_stratified(frames[split], CONFIG["smoke_samples_per_class"])
        CONFIG["batch_size"] = min(CONFIG["batch_size"], 8)
    print("Dữ liệu sẵn sàng. Chó và DTD chỉ dành cho đánh giá cuối.", flush=True)
    return frames, audit, provenance, root, api


if AUTO_RUN:
    PREPARED_DATA = prepare_data()


## Xây dựng CNN baseline, MobileNetV2, EfficientNet-B0 và ResNet50


In [ ]:
def ensure_pretrained_weights(name):
    from urllib.request import urlopen

    weights = {
        "resnet50": models.ResNet50_Weights.IMAGENET1K_V1,
        "mobilenet_v2": models.MobileNet_V2_Weights.IMAGENET1K_V1,
        "efficientnet_b0": models.EfficientNet_B0_Weights.IMAGENET1K_V1,
    }[name]
    filename = weights.url.rsplit("/", 1)[-1]
    expected = filename.rsplit("-", 1)[-1].split(".")[0]
    target = Path(torch.hub.get_dir()) / "checkpoints" / filename
    if target.is_file() and file_hash(target).startswith(expected):
        log_event("Trọng số ImageNet", "cached", model=name)
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(".download")
    started = time.monotonic()
    try:
        with urlopen(weights.url, timeout=CONFIG["network_read_timeout_seconds"]) as response:
            size = int(response.headers.get("Content-Length", 0))
            chunks = iter(lambda: response.read(1024 * 1024), b"")
            count = 0
            with temporary.open("wb") as stream:
                for _, chunk in progress(
                    chunks,
                    "Tải ImageNet " + name,
                    total=math.ceil(size / (1024 * 1024)) if size else None,
                ):
                    if time.monotonic() - started > CONFIG["download_timeout_seconds"]:
                        raise TimeoutError("Tải trọng số ImageNet quá thời gian cấu hình.")
                    stream.write(chunk)
                    count += len(chunk)
            if (size and count != size) or not file_hash(temporary).startswith(expected):
                raise ValueError("Trọng số tải thiếu hoặc không đúng checksum.")
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)


class CatClassifier(nn.Module):
    def __init__(self, name, pretrained=True, dropout=None):
        super().__init__()
        self.name = name
        if pretrained and name != "cnn_baseline":
            ensure_pretrained_weights(name)
        if name == "efficientnet_b0":
            net = models.efficientnet_b0(
                weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            )
            self.features, dimension = net.features, net.classifier[-1].in_features
        elif name == "mobilenet_v2":
            net = models.mobilenet_v2(
                weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
            )
            self.features, dimension = net.features, net.classifier[-1].in_features
        elif name == "resnet50":
            net = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
            )
            self.features, dimension = nn.Sequential(*list(net.children())[:-2]), net.fc.in_features
        elif name == "cnn_baseline":
            layers, previous = [], 3
            for channels in [32, 64, 128]:
                layers += [
                    nn.Sequential(
                        nn.Conv2d(previous, channels, 3, padding=1),
                        nn.BatchNorm2d(channels),
                        nn.ReLU(),
                        nn.MaxPool2d(2),
                    )
                ]
                previous = channels
            self.features, dimension = nn.Sequential(*layers), 128
        else:
            raise ValueError("Unknown model: " + name)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Dropout(CONFIG["dropout"] if dropout is None else dropout), nn.Linear(dimension, 12)
        )
        self.feature_dim = dimension
        self.tune_indices = []

    def forward_features(self, x):
        return self.pool(self.features(x)).flatten(1)

    def forward(self, x):
        return self.head(self.forward_features(x))

    def configure_stage(self, stage):
        for parameter in self.features.parameters():
            parameter.requires_grad = stage == "scratch"
        for parameter in self.head.parameters():
            parameter.requires_grad = True
        self.tune_indices = (
            list(range(len(self.features) - CONFIG["fine_tune_blocks"], len(self.features)))
            if stage == "finetuned"
            else []
        )
        for index in self.tune_indices:
            for parameter in self.features[index].parameters():
                parameter.requires_grad = True

    def training_mode(self, stage):
        self.train()
        if stage != "scratch":
            self.features.eval()
            for index in self.tune_indices:
                self.features[index].train()


## Chạy một epoch train hoặc validation


In [ ]:
def classification_metrics(labels, predictions):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=list(range(12)), average="macro", zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
    }


def epoch_pass(
    model,
    loader,
    criterion,
    device,
    optimizer=None,
    stage="frozen",
    batch_limit=None,
    description=None,
):
    training = optimizer is not None
    model.training_mode(stage) if training else model.eval()
    losses, labels, predictions = [], [], []
    total = len(loader) if batch_limit is None else min(len(loader), batch_limit)
    batches = loader if batch_limit is None else islice(loader, batch_limit)
    label = description or f"{model.name}/{stage}/{'train' if training else 'validation'}"
    with timed_step(label):
        for bar, (images, targets, _) in progress(batches, label, total=total):
            images, targets = images.to(device), targets.to(device)
            with torch.set_grad_enabled(training):
                logits = model(images)
                loss = criterion(logits, targets)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Loss không hữu hạn; dừng trước khi lưu checkpoint.")
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    optimizer.step()
            losses.extend([float(loss.detach())] * len(targets))
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(1).detach().cpu().tolist())
            live = classification_metrics(labels, predictions)
            bar.set_postfix(
                loss=f"{np.mean(losses):.4f}",
                acc=f"{live['accuracy']:.3f}",
                f1=f"{live['macro_f1']:.3f}",
                refresh=False,
            )
    if not labels:
        raise RuntimeError("Không có batch dữ liệu.")
    return {"loss": float(np.mean(losses)), **classification_metrics(labels, predictions)}


## Đóng gói trọng số và thông tin checkpoint


In [ ]:
def checkpoint_payload(model, config, info, stage, epoch, score):
    return {
        "model_name": model.name,
        "state_dict": model.state_dict(),
        "class_mapping": CLASS_TO_IDX,
        "config": dict(config),
        "run_id": info["run_id"],
        "split_id": info["split_id"],
        "stage": stage,
        "epoch": epoch,
        "validation_macro_f1": score,
        "git_commit": info.get("git_commit"),
        "notebook_source_sha256": info.get("notebook_source_sha256"),
    }


## Chọn checkpoint theo F1, Accuracy và epoch của QĐ9


In [ ]:
def is_better(candidate, current):
    if current is None:
        return True
    delta = candidate["macro_f1"] - current["macro_f1"]
    if abs(delta) < 1e-4:
        if candidate["accuracy"] != current["accuracy"]:
            return candidate["accuracy"] > current["accuracy"]
        return candidate["epoch"] < current["epoch"]
    return delta > 0


## Huấn luyện một giai đoạn và lưu checkpoint tốt nhất


In [ ]:
def train_stage(model, stage, train_frame, val_frame, run_dir, info, device, api):
    model.configure_stage(stage)
    if stage == "finetuned":
        groups = [
            {"params": model.head.parameters(), "lr": CONFIG["head_lr"]},
            {
                "params": [p for p in model.features.parameters() if p.requires_grad],
                "lr": CONFIG["backbone_lr"],
            },
        ]
    else:
        groups = [
            {
                "params": [p for p in model.parameters() if p.requires_grad],
                "lr": CONFIG["frozen_lr"],
            }
        ]
    optimizer = torch.optim.AdamW(groups, weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=CONFIG["scheduler_patience"]
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
    train_loader = make_loader(train_frame, training=True, seed=info["training_seed"])
    val_loader = make_loader(val_frame)
    max_epochs = (
        1
        if CONFIG["profile"] == "smoke"
        else CONFIG[
            {
                "frozen": "frozen_epochs",
                "finetuned": "finetuned_epochs",
                "scratch": "baseline_epochs",
            }[stage]
        ]
    )
    limit = CONFIG["smoke_batches"] if CONFIG["profile"] == "smoke" else None
    folder = run_dir / stage
    folder.mkdir(exist_ok=True)
    best, stale = None, 0
    for epoch in range(1, max_epochs + 1):
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        started = time.monotonic()
        epoch_learning_rate = float(optimizer.param_groups[0]["lr"])
        training = epoch_pass(
            model,
            train_loader,
            criterion,
            device,
            optimizer,
            stage,
            limit,
            description=f"{model.name}/{stage}/train {epoch}/{max_epochs}",
        )
        validation = epoch_pass(
            model,
            val_loader,
            criterion,
            device,
            batch_limit=limit,
            description=f"{model.name}/{stage}/validation {epoch}/{max_epochs}",
        )
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        scheduler.step(validation["macro_f1"])
        record = {
            "stage": stage,
            "epoch": epoch,
            "seconds": time.monotonic() - started,
            "learning_rate": epoch_learning_rate,
            **{"train_" + k: v for k, v in training.items()},
            **{"val_" + k: v for k, v in validation.items()},
        }
        info.setdefault("history", []).append(record)
        pd.DataFrame(info["history"]).to_csv(run_dir / "history.csv", index=False)
        print(json.dumps(record), flush=True)
        candidate = {**validation, "epoch": epoch}
        if is_better(candidate, best):
            best, stale = candidate, 0
            payload = checkpoint_payload(model, CONFIG, info, stage, epoch, validation["macro_f1"])
            payload["validation_accuracy"] = validation["accuracy"]
            temporary = folder / "checkpoint.pt.tmp"
            torch.save(payload, temporary)
            temporary.replace(folder / "checkpoint.pt")
            write_json(
                folder / "metrics.json",
                {
                    "selection_metric": "validation_macro_f1",
                    "value": best["macro_f1"],
                    "accuracy": best["accuracy"],
                    "macro_precision": best["macro_precision"],
                    "macro_recall": best["macro_recall"],
                    "weighted_f1": best["weighted_f1"],
                    "epoch": epoch,
                    "tie_f1_tolerance": 1e-4,
                    "unfrozen_feature_blocks": model.tune_indices,
                    "is_smoke_test": CONFIG["profile"] == "smoke",
                },
            )
        else:
            stale += 1
        log_event(
            f"{model.name}/{stage}/epoch",
            "metrics",
            **record,
            best_macro_f1=best["macro_f1"],
            stale_epochs=stale,
        )
        if stale >= CONFIG["early_stopping_patience"]:
            break
    saved = torch.load(folder / "checkpoint.pt", map_location=device, weights_only=True)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    if GROUP_SAVE_STAGE is not None:
        metrics = json.loads((folder / "metrics.json").read_text(encoding="utf-8"))
        success = GROUP_SAVE_STAGE(
            run_dir=run_dir,
            stage=stage,
            checkpoint=saved,
            metrics=metrics,
            repo_id=CONFIG["hf_repo_id"],
        )
        info["sync_status"][stage + "_checkpoint"] = bool(success)
        write_json(run_dir / "info.json", {k: v for k, v in info.items() if k != "history"})
    return folder / "checkpoint.pt"


## Nạp checkpoint độc lập để kiểm tra và suy luận


In [ ]:
def reload_checkpoint(path, device="cpu"):
    saved = torch.load(path, map_location=device, weights_only=True)
    if saved["class_mapping"] != CLASS_TO_IDX:
        raise RuntimeError("Class mapping không khớp.")
    model = CatClassifier(
        saved["model_name"], pretrained=False, dropout=saved["config"]["dropout"]
    ).to(device)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return model, saved


## Trích xuất đặc trưng, logits và lưu NPZ


In [ ]:
@torch.no_grad()
def extract_arrays(model, frame, device, split_name, checkpoint_id="", split_id="", view="full"):
    model.eval()
    features, logits, labels, identifiers = [], [], [], []
    ordered = frame.reset_index(drop=True)  # Giữ đúng thứ tự manifest đã bàn giao.
    for _, (images, targets, image_ids) in progress(
        make_loader(ordered, view=view), f"Đặc trưng {model.name}/{split_name}"
    ):
        images = images.to(device)
        h = model.forward_features(images)
        z = model.head(h)
        features.append(h.cpu().numpy())
        logits.append(z.cpu().numpy())
        labels.extend(targets.tolist())
        identifiers.extend(image_ids)
    if not identifiers:
        raise ValueError("Tập xuất rỗng: " + split_name)
    if identifiers != ordered.image_id.tolist():
        raise AssertionError("Thứ tự .npz lệch manifest.")
    return {
        "features": np.concatenate(features).astype(np.float32),
        "logits": np.concatenate(logits).astype(np.float32),
        "labels": np.asarray(labels, dtype=np.int64),
        "image_id": np.asarray(identifiers, dtype=str),
        "split": np.asarray([split_name] * len(labels), dtype=str),
        "checkpoint_id": np.asarray(checkpoint_id),
        "split_id": np.asarray(split_id),
    }


def save_npz(path, arrays):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as output:
        np.savez_compressed(output, **arrays)
    temporary.replace(path)
    with np.load(path, allow_pickle=False) as check:
        assert check["features"].shape[0] == check["logits"].shape[0] == len(check["image_id"])
        assert check["logits"].shape[1] == 12


## Tính MSP, MaxLogit và Energy


In [ ]:
def logits_scores(logits, temperature=1.0):
    logits = np.asarray(logits, dtype=np.float64)
    return {
        "MSP": softmax(logits, axis=1).max(axis=1),
        "MaxLogit": logits.max(axis=1),
        "Energy": temperature * logsumexp(logits / temperature, axis=1),
    }


## Fit và tính Mahalanobis, KNN


In [ ]:
class DistanceScores:
    def fit(self, features, labels, k=10, knn_l2=True):
        x, y = np.asarray(features, dtype=np.float64), np.asarray(labels)
        if set(y.tolist()) != set(range(12)):
            raise ValueError("Train bank phải có đủ 12 lớp.")
        if len(x) < k:
            raise ValueError("Train bank nhỏ hơn k; không tự đổi k.")
        self.means = np.stack([x[y == c].mean(axis=0) for c in range(12)])
        residuals = x - self.means[y]
        covariance = LedoitWolf().fit(residuals)
        self.precision, self.shrinkage = covariance.precision_, float(covariance.shrinkage_)
        self.k, self.knn_l2 = k, knn_l2
        self.bank = self.normalize(x) if knn_l2 else x
        self.neighbors = NearestNeighbors(
            n_neighbors=k, metric="euclidean", algorithm="brute", n_jobs=2
        ).fit(self.bank)
        return self

    @staticmethod
    def normalize(x):
        return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

    def score(self, features):
        x = np.asarray(features, dtype=np.float64)
        xp, mp = x @ self.precision, self.means @ self.precision
        squared = (
            (xp * x).sum(axis=1)[:, None]
            - 2 * xp @ self.means.T
            + (mp * self.means).sum(axis=1)[None, :]
        )
        query = self.normalize(x) if self.knn_l2 else x
        distances = self.neighbors.kneighbors(query, return_distance=True)[0]
        return {
            "Mahalanobis": -np.maximum(squared, 0).min(axis=1),
            "KNN": -distances[:, self.k - 1],
        }


## Chốt ngưỡng validation và tính các chỉ số OOD


In [ ]:
def all_scores(arrays, distance):
    return {
        **logits_scores(arrays["logits"], temperature=1.0),
        **distance.score(arrays["features"]),
    }


def locked_thresholds(validation_scores):
    # ID validation, không dùng near/far-OOD. NumPy linear quantile được ghi rõ trong artifact.
    return {
        name: float(np.quantile(values, 0.05, method="linear"))
        for name, values in validation_scores.items()
    }


def ood_metrics(id_scores, out_scores, threshold):
    y = np.r_[np.ones(len(id_scores)), np.zeros(len(out_scores))]
    s = np.r_[id_scores, out_scores]
    fpr, tpr, _ = roc_curve(y, s, drop_intermediate=False)
    # Đọc bao ROC: max TPR tại mỗi FPR rồi nội suy FPR theo TPR; cùng quy tắc cho mọi phương pháp.
    unique_tpr = np.unique(tpr)
    min_fpr = np.array([fpr[tpr == value].min() for value in unique_tpr])
    return {
        "auroc": float(roc_auc_score(y, s)),
        "fpr_at_95_tpr": float(np.interp(0.95, unique_tpr, min_fpr)),
        "locked_threshold": float(threshold),
        "tpr_locked": float(np.mean(id_scores >= threshold)),
        "fpr_locked": float(np.mean(out_scores >= threshold)),
        "n_id": len(id_scores),
        "n_ood": len(out_scores),
    }


def accuracy_coverage(arrays, scores):
    correct = arrays["logits"].argmax(1) == arrays["labels"]
    order = np.argsort(-np.asarray(scores), kind="stable")
    sorted_scores = np.asarray(scores)[order]
    # Chỉ báo điểm cuối mỗi nhóm hòa; không tùy ý giữ một phần nhóm cùng điểm.
    ends = np.r_[np.flatnonzero(sorted_scores[:-1] != sorted_scores[1:]), len(order) - 1]
    accepted = ends + 1
    return pd.DataFrame(
        {
            "threshold": sorted_scores[ends],
            "coverage": accepted / len(order),
            "accuracy": np.cumsum(correct[order])[ends] / accepted,
            "accepted": accepted,
        }
    )


## Tính khoảng tin cậy AUROC và chênh lệch bằng bootstrap ghép cặp


In [ ]:
def paired_bootstrap(id_scores, out_scores, split, repeats=None):
    names = list(id_scores)
    n_id, n_out = len(id_scores[names[0]]), len(out_scores[names[0]])
    y = np.r_[np.ones(n_id), np.zeros(n_out)]
    values = {name: np.r_[id_scores[name], out_scores[name]] for name in names}
    repeats = CONFIG["bootstrap_repeats"] if repeats is None else repeats
    rng = np.random.default_rng(CONFIG["bootstrap_seed"])
    samples = np.empty((repeats, len(names)))
    for _, b in progress(range(repeats), "Bootstrap " + split):
        index = np.r_[rng.integers(n_id, size=n_id), n_id + rng.integers(n_out, size=n_out)]
        samples[b] = [roc_auc_score(y, values[name][index]) for name in names]
    point = {name: float(roc_auc_score(y, values[name])) for name in names}
    rows = []
    for name, column in zip(names, samples.T):
        low, high = np.quantile(column, [0.025, 0.975])
        rows.append(
            {
                "ood_split": split,
                "comparison": name,
                "estimate": point[name],
                "ci_low": float(low),
                "ci_high": float(high),
                "bootstrap_repeats": repeats,
            }
        )
    for distance in ["Mahalanobis", "KNN"]:
        for logit in ["MSP", "MaxLogit", "Energy"]:
            delta = samples[:, names.index(distance)] - samples[:, names.index(logit)]
            low, high = np.quantile(delta, [0.025, 0.975])
            rows.append(
                {
                    "ood_split": split,
                    "comparison": distance + " - " + logit,
                    "estimate": point[distance] - point[logit],
                    "ci_low": float(low),
                    "ci_high": float(high),
                    "bootstrap_repeats": repeats,
                    "hypothesis": "distance > logit" if split == "near_ood" else "descriptive only",
                }
            )
    return pd.DataFrame(rows)


## Fit temperature và tính ECE


In [ ]:
def temperature_fit(logits, labels):
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=int)

    def nll(log_t):
        z = logits / np.exp(log_t)
        return float(np.mean(logsumexp(z, axis=1) - z[np.arange(len(z)), labels]))

    optimum = minimize_scalar(nll, bounds=(-4, 4), method="bounded")
    if not optimum.success:
        raise RuntimeError("Temperature optimizer failed.")
    return float(np.exp(optimum.x))


def calibration_metrics(logits, labels, temperature=1.0):
    probabilities = softmax(np.asarray(logits) / temperature, axis=1)
    confidence, prediction = probabilities.max(1), probabilities.argmax(1)
    rows, ece = [], 0.0
    edges = np.linspace(0, 1, CONFIG["ece_bins"] + 1)
    bins = np.minimum(np.searchsorted(edges, confidence, side="right") - 1, CONFIG["ece_bins"] - 1)
    for b in range(CONFIG["ece_bins"]):
        chosen = bins == b
        n = int(chosen.sum())
        if not n:
            continue
        accuracy, mean_confidence = float(np.mean(prediction[chosen] == labels[chosen])), float(
            confidence[chosen].mean()
        )
        ece += n / len(labels) * abs(accuracy - mean_confidence)
        rows.append({"bin": b, "n": n, "accuracy": accuracy, "confidence": mean_confidence})
    return float(ece), pd.DataFrame(rows)


## Vẽ reliability diagram và đường cong huấn luyện


In [ ]:
def reliability_plot(before, after, destination, title="Reliability diagram"):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
    for axis, frame, label in zip(axes, [before, after], ["Before scaling", "After scaling"]):
        axis.plot([0, 1], [0, 1], linestyle="--", color="gray")
        axis.plot(frame.confidence, frame.accuracy, marker="o")
        axis.set(xlabel="Mean confidence", ylabel="Accuracy", title=label, xlim=(0, 1), ylim=(0, 1))
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(destination)
    plt.close(fig)


def training_plot(history, destination):
    frame = pd.DataFrame(history)
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for axis, metric in zip(
        axes.flat, ["loss", "accuracy", "macro_precision", "macro_recall", "macro_f1"]
    ):
        for split in ["train", "val"]:
            axis.plot(np.arange(1, len(frame) + 1), frame[split + "_" + metric], label=split)
        axis.set(xlabel="Epoch across stages", ylabel=metric)
        axis.legend()
    axes.flat[-1].plot(np.arange(1, len(frame) + 1), frame.learning_rate)
    axes.flat[-1].set(xlabel="Epoch across stages", ylabel="Head learning rate", yscale="log")
    for boundary in np.flatnonzero(frame.stage.to_numpy()[1:] != frame.stage.to_numpy()[:-1]) + 1.5:
        for axis in axes.flat:
            axis.axvline(boundary, color="gray", linestyle="--", alpha=0.5)
    fig.tight_layout()
    fig.savefig(destination)
    plt.close(fig)


## Tạo heatmap Grad-CAM


In [ ]:
def gradcam(model, tensor, class_index=None):
    model.eval()
    activations = []
    handle = model.features[-1].register_forward_hook(
        lambda module, inputs, output: activations.append(output)
    )
    try:
        x = tensor.detach().clone().requires_grad_(True)
        logits = model(x)
        if class_index is None:
            class_index = int(logits.argmax(1).item())
        activation = activations[0]
        gradient = torch.autograd.grad(logits[0, class_index], activation)[0]
        heat = torch.relu((gradient.mean((2, 3), keepdim=True) * activation).sum(1, keepdim=True))
        heat = nn.functional.interpolate(
            heat, size=tensor.shape[-2:], mode="bilinear", align_corners=False
        )[0, 0]
        heat = heat - heat.min()
        span = heat.max()
        heat = heat / span if span > 0 else torch.zeros_like(heat)
        return heat.detach().cpu().numpy(), class_index
    finally:
        handle.remove()


## Tính IoU, tỷ lệ năng lượng R và tỷ lệ diện tích f theo QĐ5


In [ ]:
def cam_overlap(heat, trimap, threshold=0.5):
    if heat.shape != trimap.shape or not np.isfinite(heat).all():
        raise ValueError("Heatmap/trimap sai kích thước hoặc có giá trị không hữu hạn.")
    valid = trimap != 3
    if not valid.any():
        raise ValueError("Trimap không có điểm ảnh hợp lệ.")
    foreground = trimap == 1
    f = float(foreground.sum() / valid.sum())
    degenerate = bool(heat.max() == heat.min())
    attention = (heat >= threshold) & valid
    union = np.logical_or(attention, foreground).sum()
    iou = (
        0.0
        if degenerate or not union
        else float(np.logical_and(attention, foreground).sum() / union)
    )
    total = float(heat[valid].sum())
    ratio = None if degenerate or total == 0 else float(heat[foreground].sum() / total)
    return {"iou": iou, "R": ratio, "f": f, "degenerate": degenerate}


## Báo cáo Grad-CAM, mIoU, R và phân tích lỗi


In [ ]:
def gradcam_report(model, frame, device, folder, split="validation", iou_boundary=None):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    valid = frame[frame.has_valid_trimap].sort_values("image_id")
    if split == "validation" and CONFIG["profile"] == "smoke":
        valid = small_stratified(valid, 1).head(2)
    if valid.empty:
        summary = {
            "split": split,
            "status": "unavailable",
            "n_evaluated": 0,
            "iou_boundary": iou_boundary,
        }
        write_json(folder / "gradcam_summary.json", summary)
        return summary
    rows = []
    for _, (_, row) in progress(
        valid.iterrows(), f"Grad-CAM {model.name}/{split}", total=len(valid)
    ):
        image = read_rgb(row.image_path)
        tensor = make_transform()(image).unsqueeze(0).to(device)
        heat, prediction = gradcam(model, tensor)
        with Image.open(row.trimap_path) as source:
            mask = np.array(
                source.resize(
                    (CONFIG["image_size"], CONFIG["image_size"]), Image.Resampling.NEAREST
                )
            )
        scores = {theta: cam_overlap(heat, mask, theta) for theta in [0.3, 0.5, 0.7]}
        rows.append(
            {
                "image_id": row.image_id,
                "label": int(row.label),
                "prediction": prediction,
                "correct": prediction == row.label,
                **scores[0.5],
                "iou_0.3": scores[0.3]["iou"],
                "iou_0.7": scores[0.7]["iou"],
            }
        )
        if len(rows) <= 4:
            fig, axis = plt.subplots()
            axis.imshow(image.resize((CONFIG["image_size"], CONFIG["image_size"])))
            axis.imshow(heat, cmap="magma", alpha=0.4, vmin=0, vmax=1)
            axis.set_title(row.image_id)
            axis.axis("off")
            fig.savefig(folder / (row.image_id + ".png"), bbox_inches="tight")
            plt.close(fig)
    result = pd.DataFrame(rows)
    if split == "validation":
        correct = result.loc[result.correct, "iou"]
        iou_boundary = float(correct.median()) if len(correct) else None
    result["error_group"] = [
        (
            "correct"
            if row.correct
            else (
                "unavailable_boundary"
                if iou_boundary is None
                else ("low_iou" if row.iou < iou_boundary else "high_iou")
            )
        )
        for row in result.itertuples()
    ]
    result.to_csv(folder / "gradcam_per_image.csv", index=False)

    def mean(values):
        return float(values.dropna().mean()) if values.notna().any() else None

    summary = {
        "split": split,
        "n_evaluated": len(result),
        "excluded_invalid_masks": int((~frame.has_valid_trimap).sum()),
        "miou": mean(result.iou),
        "miou_correct": mean(result.loc[result.correct, "iou"]),
        "miou_incorrect": mean(result.loc[~result.correct, "iou"]),
        "miou_0.3": mean(result["iou_0.3"]),
        "miou_0.7": mean(result["iou_0.7"]),
        "mean_R": mean(result.R),
        "mean_f": mean(result.f),
        "R_defined_count": int(result.R.notna().sum()),
        "mean_f_on_R_defined": mean(result.loc[result.R.notna(), "f"]),
        "R_above_uniform_fraction": (
            float((result.loc[result.R.notna(), "R"] > result.loc[result.R.notna(), "f"]).mean())
            if result.R.notna().any()
            else None
        ),
        "degenerate_count": int(result.degenerate.sum()),
        "iou_boundary": iou_boundary,
        "boundary_source": "correct ID validation, theta=0.5",
        "R_definition": "CAM energy on label 1 / CAM energy on labels 1 and 2",
        "f_definition": "label 1 area / labels 1 and 2 area",
        "is_smoke_test": CONFIG["profile"] == "smoke",
    }
    write_json(folder / "gradcam_summary.json", summary)
    fig, axis = plt.subplots()
    for correct, hatch, label in [(True, "", "Correct"), (False, "//", "Incorrect")]:
        axis.hist(
            result.loc[result.correct == correct, "iou"],
            bins=np.linspace(0, 1, 11),
            alpha=0.5,
            edgecolor="black",
            hatch=hatch,
            label=label,
        )
    axis.set(xlabel="IoU", ylabel="Images")
    axis.legend()
    fig.savefig(folder / "iou_distribution.png")
    plt.close(fig)
    fig, axis = plt.subplots()
    axis.plot([0, 1], [0, 1], "--", color="gray", label="Uniform reference R=f")
    for correct, marker, label in [(True, "o", "Correct"), (False, "x", "Incorrect")]:
        chosen = result[result.correct.eq(correct) & result.R.notna()]
        axis.scatter(chosen.f, chosen.R, marker=marker, alpha=0.6, label=label)
    axis.set(
        xlabel="Foreground area fraction f",
        ylabel="CAM energy fraction R",
        xlim=(0, 1),
        ylim=(0, 1),
    )
    axis.legend()
    fig.savefig(folder / "R_vs_f.png")
    plt.close(fig)
    log_event("Grad-CAM " + split, "metrics", **summary)
    return summary


## Đánh giá phân loại và confusion matrix


In [ ]:
def classification_outputs(arrays, folder):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    y, p = arrays["labels"], arrays["logits"].argmax(1)
    write_json(
        folder / "classification.json",
        classification_report(
            y, p, labels=list(range(12)), target_names=CAT_BREEDS, output_dict=True, zero_division=0
        ),
    )
    pd.DataFrame(
        {
            "image_id": arrays["image_id"],
            "true_label": y,
            "predicted_label": p,
            "confidence_T1": softmax(arrays["logits"], axis=1).max(1),
        }
    ).to_csv(folder / "predictions.csv", index=False)
    matrix = confusion_matrix(y, p, labels=list(range(12)))
    pd.DataFrame(matrix, index=CAT_BREEDS, columns=CAT_BREEDS).to_csv(
        folder / "confusion_matrix.csv"
    )
    fig, axis = plt.subplots(figsize=(9, 8))
    graphic = axis.imshow(matrix, cmap="Blues")
    axis.set_xticks(range(12), CAT_BREEDS, rotation=90)
    axis.set_yticks(range(12), CAT_BREEDS)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
    fig.colorbar(graphic, ax=axis)
    fig.tight_layout()
    fig.savefig(folder / "confusion_matrix.png")
    plt.close(fig)
    report = classification_report(
        y, p, labels=list(range(12)), target_names=CAT_BREEDS, output_dict=True, zero_division=0
    )
    pd.DataFrame({name: row for name, row in report.items() if isinstance(row, dict)}).T.to_csv(
        folder / "classification_report.csv"
    )
    summary = classification_metrics(y, p)
    write_json(folder / "summary.json", summary)
    log_event(str(folder.parent.name) + "/" + folder.name, "metrics", **summary)
    print(
        classification_report(
            y, p, labels=list(range(12)), target_names=CAT_BREEDS, digits=4, zero_division=0
        ),
        flush=True,
    )
    return summary


## Dự đoán một ảnh và áp dụng ngưỡng từ chối


In [ ]:
def load_inference_bundle(stage_dir, device="cpu", allow_provisional=False):
    stage_dir = Path(stage_dir)
    model, saved = reload_checkpoint(stage_dir / "checkpoint.pt", device)
    thresholds = json.loads((stage_dir / "thresholds.json").read_text(encoding="utf-8"))
    if thresholds["checkpoint_id"] != file_hash(stage_dir / "checkpoint.pt"):
        raise ValueError("Ngưỡng không thuộc checkpoint này.")
    if not allow_provisional and (thresholds["provisional"] or thresholds["is_smoke_test"]):
        raise ValueError("Chỉ dùng ngưỡng đã khóa và checkpoint chính thức để triển khai.")
    with np.load(stage_dir / "distance_parameters.npz", allow_pickle=False) as arrays:
        distance = DistanceScores()
        distance.means, distance.precision = arrays["means"], arrays["precision"]
        distance.k, distance.knn_l2 = int(arrays["knn_k"]), bool(arrays["knn_l2"])
        distance.bank = (
            distance.normalize(arrays["train_features"])
            if distance.knn_l2
            else arrays["train_features"]
        )
        distance.neighbors = NearestNeighbors(
            n_neighbors=distance.k, metric="euclidean", algorithm="brute", n_jobs=2
        ).fit(distance.bank)
    calibration = json.loads((stage_dir / "calibration.json").read_text(encoding="utf-8"))
    return {
        "model": model,
        "distance": distance,
        "thresholds": thresholds["thresholds"],
        "temperature": calibration["temperature"],
        "config": saved["config"],
        "device": device,
    }


def predict_image(bundle, image_path, method="Energy", show_cam=True):
    tensor = (
        make_transform(config=bundle["config"])(read_rgb(image_path))
        .unsqueeze(0)
        .to(bundle["device"])
    )
    model = bundle["model"].eval()
    with torch.no_grad():
        features = model.forward_features(tensor)
        logits = model.head(features).cpu().numpy()
    scores = all_scores({"features": features.cpu().numpy(), "logits": logits}, bundle["distance"])
    index = int(logits.argmax(1)[0])
    threshold = bundle["thresholds"][method]
    accepted = bool(scores[method][0] >= threshold)
    result = {
        "label": CAT_BREEDS[index] if accepted else "không xác định",
        "predicted_class_before_rejection": CAT_BREEDS[index],
        "accepted": accepted,
        "method": method,
        "score": float(scores[method][0]),
        "threshold": threshold,
        "confidence": float(softmax(logits / bundle["temperature"], axis=1).max()),
        "is_smoke_test": bundle["config"]["profile"] == "smoke",
    }
    if show_cam:
        result["heatmap"] = gradcam(model, tensor, index)[0]
    return result


## So sánh ảnh đầy đủ với biến thể trên cùng các ảnh hợp lệ


In [ ]:
def evaluate_views(model, test_frame, device, stage_dir, checkpoint_id, split_id):
    views = []
    if CONFIG["run_background_ablation"]:
        views.append(("no_background", "has_valid_trimap"))
    if CONFIG["run_head_ablation"]:
        views.append(("head", "has_bbox"))
    for view, available in views:
        cohort = test_frame[test_frame[available]]
        if cohort.empty:
            write_json(
                stage_dir / (view + "_comparison.json"),
                {
                    "status": "unavailable",
                    "n": 0,
                    "reason": "Tập test chính thức không có annotation phù hợp; không tự tạo nhãn.",
                },
            )
            continue
        full = extract_arrays(
            model, cohort, device, "test_" + view + "_cohort", checkpoint_id, split_id
        )
        changed = extract_arrays(
            model, cohort, device, "test_" + view, checkpoint_id, split_id, view=view
        )
        if not np.array_equal(full["image_id"], changed["image_id"]):
            raise AssertionError("Hai biến thể phải dùng cùng tập ảnh và thứ tự.")
        write_json(
            stage_dir / (view + "_comparison.json"),
            {
                "status": "completed",
                "n": len(cohort),
                "excluded": len(test_frame) - len(cohort),
                "scope": "inference-only, no retraining",
                "full": classification_outputs(full, stage_dir / (view + "_full")),
                "changed": classification_outputs(changed, stage_dir / view),
            },
        )


## Đánh giá ID test và OOD sau khi khóa giao thức


In [ ]:
def evaluate_final(
    model, frames, distance, thresholds, temperature, device, stage_dir, checkpoint_id, split_id
):
    if not (CONFIG["evaluation_locked"] and CONFIG["protocol_confirmed"]):
        raise RuntimeError("Chưa khóa giao thức/validation, không đánh giá test.")
    test = extract_arrays(model, frames["test"], device, "test", checkpoint_id, split_id)
    save_npz(stage_dir / "test.npz", test)
    classification_outputs(test, stage_dir / "test_classification")
    test_scores = all_scores(test, distance)
    rows = []
    confidence = []
    for split in ["near_ood", "far_ood"]:
        if split not in frames or frames[split].empty:
            print("Chưa có", split, "— không bịa kết quả.", flush=True)
            continue
        outside = extract_arrays(model, frames[split], device, split, checkpoint_id, split_id)
        save_npz(stage_dir / (split + ".npz"), outside)
        out_scores = all_scores(outside, distance)
        confidence.append(paired_bootstrap(test_scores, out_scores, split))
        for method in test_scores:
            rows.append(
                {
                    "ood_split": split,
                    "method": method,
                    **ood_metrics(test_scores[method], out_scores[method], thresholds[method]),
                }
            )
    pd.DataFrame(rows).to_csv(stage_dir / "ood_results.csv", index=False)
    for result in rows:
        log_event("OOD test", "metrics", **result)
    if confidence:
        pd.concat(confidence, ignore_index=True).to_csv(
            stage_dir / "ood_paired_confidence.csv", index=False
        )
    validation_cam = json.loads(
        (stage_dir / "gradcam/gradcam_summary.json").read_text(encoding="utf-8")
    )
    gradcam_report(
        model,
        frames["test"],
        device,
        stage_dir / "test_gradcam",
        split="test",
        iou_boundary=validation_cam["iou_boundary"],
    )
    fig, axis = plt.subplots()
    for method, scores in test_scores.items():
        curve = accuracy_coverage(test, scores)
        curve.to_csv(stage_dir / ("accuracy_coverage_" + method + ".csv"), index=False)
        axis.plot(curve.coverage, curve.accuracy, label=method)
    axis.set(xlabel="Coverage", ylabel="Accuracy on accepted ID")
    axis.legend()
    fig.tight_layout()
    fig.savefig(stage_dir / "accuracy_coverage.png")
    plt.close(fig)
    before, bins_before = calibration_metrics(test["logits"], test["labels"])
    after, bins_after = calibration_metrics(test["logits"], test["labels"], temperature)
    write_json(
        stage_dir / "test_calibration.json",
        {"temperature": temperature, "ece_before": before, "ece_after": after},
    )
    bins_before.to_csv(stage_dir / "reliability_before.csv", index=False)
    bins_after.to_csv(stage_dir / "reliability_after.csv", index=False)
    reliability_plot(
        bins_before, bins_after, stage_dir / "test_reliability.png", "Held-out ID test"
    )
    if CONFIG["run_background_ablation"] or CONFIG["run_head_ablation"]:
        evaluate_views(model, frames["test"], device, stage_dir, checkpoint_id, split_id)
    if CONFIG["run_ood_sensitivity"]:
        sensitivity = []
        outside = extract_arrays(
            model, frames["near_ood"], device, "near_ood", checkpoint_id, split_id
        )
        for l2 in [False, True]:
            for k in [1, 5, 10, 20, 50]:
                if k > len(distance.bank):
                    continue
                estimator = NearestNeighbors(n_neighbors=k, n_jobs=2).fit(
                    distance.normalize(frames["_train_arrays"]["features"])
                    if l2
                    else frames["_train_arrays"]["features"]
                )

                def knn_score(array):
                    x = array["features"]
                    x = distance.normalize(x) if l2 else x
                    return -estimator.kneighbors(x, return_distance=True)[0][:, k - 1]

                val_score = knn_score(frames["_val_arrays"])
                tau = float(np.quantile(val_score, 0.05, method="linear"))
                id_score = knn_score(test)
                sensitivity.append(
                    {"k": k, "l2": l2, **ood_metrics(id_score, knn_score(outside), tau)}
                )
        pd.DataFrame(sensitivity).to_csv(
            stage_dir / "knn_sensitivity_descriptive_only.csv", index=False
        )


## Đánh giá validation và xuất NPZ, OOD, calibration, Grad-CAM cho một checkpoint


In [ ]:
def evaluate_stage(model, frames, device, stage_dir, checkpoint_id, split_id):
    validation = extract_arrays(model, frames["val"], device, "val", checkpoint_id, split_id)
    classification_outputs(validation, stage_dir / "validation_classification")
    if model.name == "cnn_baseline":
        if CONFIG["run_final_evaluation"]:
            test = extract_arrays(model, frames["test"], device, "test", checkpoint_id, split_id)
            classification_outputs(test, stage_dir / "test_classification")
        return
    training = extract_arrays(model, frames["train"], device, "train", checkpoint_id, split_id)
    save_npz(stage_dir / "train.npz", training)
    save_npz(stage_dir / "val.npz", validation)
    distance = run_step(
        "Fit Mahalanobis và KNN",
        DistanceScores().fit,
        training["features"],
        training["labels"],
        k=10,
        knn_l2=True,
    )
    scores = all_scores(validation, distance)
    thresholds = locked_thresholds(scores)
    np.savez_compressed(
        stage_dir / "distance_parameters.npz",
        means=distance.means,
        precision=distance.precision,
        train_features=training["features"],
        train_labels=training["labels"],
        knn_k=10,
        knn_l2=True,
    )
    pd.DataFrame({"image_id": validation["image_id"], **scores}).to_csv(
        stage_dir / "validation_scores.csv", index=False
    )
    write_json(
        stage_dir / "thresholds.json",
        {
            "thresholds": thresholds,
            "quantile": 0.05,
            "quantile_method": "linear",
            "selected_from": "ID validation only",
            "checkpoint_id": checkpoint_id,
            "provisional": not CONFIG["evaluation_locked"],
            "is_smoke_test": CONFIG["profile"] == "smoke",
        },
    )
    temperature = run_step(
        "Fit temperature trên validation",
        temperature_fit,
        validation["logits"],
        validation["labels"],
    )
    before, bins_before = calibration_metrics(validation["logits"], validation["labels"])
    after, bins_after = calibration_metrics(validation["logits"], validation["labels"], temperature)
    write_json(
        stage_dir / "calibration.json",
        {
            "temperature": temperature,
            "val_ece_before": before,
            "val_ece_after": after,
            "note": "Validation đã fit T; đây là chẩn đoán in-sample, không phải test. OOD chính vẫn dùng T=1.",
        },
    )
    log_event(
        "Calibration validation",
        "metrics",
        temperature=temperature,
        ece_before=before,
        ece_after=after,
    )
    bins_after.to_csv(stage_dir / "validation_reliability.csv", index=False)
    reliability_plot(
        bins_before,
        bins_after,
        stage_dir / "validation_reliability.png",
        "Validation calibration diagnostic",
    )
    gradcam_report(model, frames["val"], device, stage_dir / "gradcam")
    if CONFIG["run_final_evaluation"]:
        evaluate_final(
            model,
            {**frames, "_train_arrays": training, "_val_arrays": validation},
            distance,
            thresholds,
            temperature,
            device,
            stage_dir,
            checkpoint_id,
            split_id,
        )


## Điều phối một model, một seed và xuất hồ sơ kết quả


In [ ]:
def train_one(model_name, seed, frames, audit, provenance, root, api, device):
    seed_everything(seed)
    started = time.monotonic()
    run_id = (
        datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        + "_"
        + model_name
        + "_s"
        + str(seed)
        + "_"
        + uuid4().hex[:8]
    )
    run_dir = Path(root) / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    info = {
        "run_id": run_id,
        "owner": CONFIG.get("hf_authenticated_owner", CONFIG["owner"]),
        "display_name": CONFIG["owner"],
        "started_at_utc": utc_now(),
        "status": "running",
        "is_smoke_test": CONFIG["profile"] == "smoke",
        "model": model_name,
        "training_seed": seed,
        "split_id": audit["split_id"],
        "data_version": provenance,
        "protocol_id": CONFIG["protocol_id"],
        "git_commit": SUBMISSION.get("git_commit"),
        "notebook_source_sha256": SUBMISSION.get("notebook_source_sha256"),
        "runner_provenance": dict(RUNNER_PROVENANCE),
        "source_submission": SUBMISSION,
        "sync_status": {},
    }
    write_json(run_dir / "class_mapping.json", CLASS_TO_IDX)
    write_json(run_dir / "config.yaml", CONFIG)
    write_json(run_dir / "info.json", info)
    (run_dir / "requirements_freeze.txt").write_text(
        run_command(
            [sys.executable, "-m", "pip", "freeze"], "Ghi phiên bản thư viện", capture=True
        ).stdout,
        encoding="utf-8",
    )
    for name, frame in frames.items():
        if not frame.empty:
            frame.to_csv(run_dir / (name + "_manifest.csv"), index=False)
    try:
        model = run_step(
            "Khởi tạo " + model_name, CatClassifier, model_name, pretrained=CONFIG["pretrained"]
        ).to(device)
        for stage in (["scratch"] if model_name == "cnn_baseline" else ["frozen", "finetuned"]):
            log_event(
                model_name + "/" + stage, "started", seed=seed, device=str(device), run_id=run_id
            )
            checkpoint = train_stage(
                model, stage, frames["train"], frames["val"], run_dir, info, device, api
            )
            restored, _ = reload_checkpoint(checkpoint, device)
            probe = next(iter(make_loader(frames["val"])))[0][:1].to(device)
            with torch.no_grad():
                if not torch.allclose(model.eval()(probe), restored(probe), rtol=1e-4, atol=1e-5):
                    raise RuntimeError("Checkpoint độc lập cho logits khác.")
            model = restored
            evaluate_stage(
                model, frames, device, checkpoint.parent, file_hash(checkpoint), audit["split_id"]
            )
            info["sync_status"][stage] = sync_run(run_dir, api, stage)
            write_json(run_dir / "info.json", {k: v for k, v in info.items() if k != "history"})
            if device.type == "cuda":
                torch.cuda.empty_cache()
        training_plot(info["history"], run_dir / "training_curves.png")
        info["status"] = "completed"
    except Exception:
        info["status"] = "failed"
        (run_dir / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")
        raise
    finally:
        info["finished_at_utc"] = utc_now()
        info["training_validation_seconds"] = sum(row["seconds"] for row in info.get("history", []))
        info["seconds_before_final_sync"] = time.monotonic() - started
        write_json(run_dir / "info.json", {k: v for k, v in info.items() if k != "history"})
        final_ok = sync_run(run_dir, api, "final" if info["status"] == "completed" else "failed")
        write_json(
            run_dir / "sync_receipt.json",
            {
                "final_upload_success": final_ok,
                "checked_at_utc": utc_now(),
                "total_seconds": time.monotonic() - started,
            },
        )
    print("Run:", run_id, "|", info["status"], "| đồng bộ:", final_ok, flush=True)
    return run_dir


## Điều phối toàn bộ pipeline theo cấu hình đã chọn


In [ ]:
def run_pipeline(prepared=None):
    official = CONFIG["profile"] == "official"
    if CONFIG["profile"] not in ["smoke", "official"]:
        raise ValueError("profile không hợp lệ.")
    if (
        CONFIG["label_smoothing"] != 0
        or CONFIG["energy_temperature"] != 1
        or CONFIG["knn_k"] != 10
        or CONFIG["fine_tune_blocks"] != 2
    ):
        raise ValueError("QĐ2/QĐ8: label smoothing=0, Energy T=1; KNN k=10; mở 2 block cuối.")
    if any(seed not in [42, 43, 44] for seed in CONFIG["training_seeds"]):
        raise ValueError("QĐ4 chỉ dùng seed 42, 43, 44.")
    if CONFIG["run_final_evaluation"] and not (
        official and CONFIG["evaluation_locked"] and CONFIG["protocol_confirmed"]
    ):
        raise RuntimeError("Test/OOD chỉ chạy sau khi khóa giao thức và lựa chọn trên validation.")
    if official:
        if (
            not CONFIG["protocol_confirmed"]
            or not torch.cuda.is_available()
            or CONFIG["image_size"] != 224
        ):
            raise RuntimeError("Official cần QĐ1–QĐ10 đã chốt, GPU và ảnh 224×224.")
        if not CONFIG["pretrained"] and any(name != "cnn_baseline" for name in CONFIG["models"]):
            raise RuntimeError("Backbone học chuyển giao cần trọng số ImageNet.")
        if not SUBMISSION.get("git_commit") or not SUBMISSION.get("notebook_source_sha256"):
            raise RuntimeError("Official cần commit và SHA source của notebook đã commit.")
    frames, audit, provenance, root, api = prepare_data() if prepared is None else prepared
    if official and (
        SUBMISSION.get("git_dirty", True)
        or not RUNNER_PROVENANCE.get("notebook_source_matches_commit")
    ):
        raise RuntimeError("Source notebook phải khớp commit đã push trước khi chạy chính thức.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    runs = [
        train_one(name, seed, frames, audit, provenance, root, api, device)
        for name in CONFIG["models"]
        for seed in CONFIG["training_seeds"]
    ]
    rows = []
    for run in runs:
        info = json.loads((run / "info.json").read_text(encoding="utf-8"))
        for stage in ["scratch", "frozen", "finetuned"]:
            metrics = run / stage / "metrics.json"
            if metrics.exists():
                values = json.loads(metrics.read_text(encoding="utf-8"))
                rows.append(
                    {
                        "model": info["model"],
                        "seed": info["training_seed"],
                        "stage": stage,
                        "val_macro_f1": values["value"],
                        "val_accuracy": values["accuracy"],
                        "val_macro_precision": values["macro_precision"],
                        "val_macro_recall": values["macro_recall"],
                        "val_weighted_f1": values["weighted_f1"],
                        "epoch": values["epoch"],
                        "run_id": info["run_id"],
                        "is_smoke_test": info["is_smoke_test"],
                    }
                )
    summary = pd.DataFrame(rows)
    summary.to_csv(root / "validation_summary.csv", index=False)
    summary.groupby(["model", "stage"])[
        [
            "val_macro_f1",
            "val_accuracy",
            "val_macro_precision",
            "val_macro_recall",
            "val_weighted_f1",
        ]
    ].agg(["count", "mean", "std"]).to_csv(root / "validation_mean_std.csv")
    best, selected = None, None
    for row in rows:
        candidate = {
            "macro_f1": row["val_macro_f1"],
            "accuracy": row["val_accuracy"],
            "epoch": row["epoch"],
        }
        if is_better(candidate, best):
            best, selected = candidate, row
    write_json(
        root / "validation_candidate.json",
        {
            "candidate": selected,
            "selection": "QĐ9, ID validation only",
            "deployment_locked": CONFIG["evaluation_locked"],
        },
    )
    print("Hoàn thành:", [run.name for run in runs], "| profile:", CONFIG["profile"], flush=True)
    print(
        "Test/OOD:",
        "đã bật sau khóa" if CONFIG["run_final_evaluation"] else "chưa chạy theo lịch tuần 10",
        flush=True,
    )
    return runs


## Thực thi phần bổ sung


In [ ]:
if AUTO_RUN:
    RUN_DIRECTORIES = run_pipeline(PREPARED_DATA)
